In [ ]:
import sys
sys.path.append('../../utils')
from functions import * 

In [ ]:
from importlib import reload
import sys
import scipy.stats as stats

# Path to the Leaflet repository
PATH_TO_LEAFLET_REPO = '/gpfs/commons/home/kisaev/Leaflet/src/beta-binomial-mix/'
sys.path.append(PATH_TO_LEAFLET_REPO)

In [ ]:
import cell_state_asign_consistency
reload  (cell_state_asign_consistency)

In [ ]:
import betabinomo_mix_singlecells
reload (betabinomo_mix_singlecells)

In [ ]:
# reload load_cluster_data
import load_cluster_data
reload (load_cluster_data)

In [ ]:
from importlib import reload
#from load_cluster_data import load_cluster_data
from betabinomo_mix_singlecells import *
#reload(betabinomo_mix_singlecells)
from cell_state_asign_consistency import *
#reload(cell_state_asign_consistency)
import torch
import sklearn.manifold 
import plotnine as p9
import time
# indicate plot should be small 4 by 4
import plotnine as p9
from plotnine import ggplot, geom_point, aes, stat_smooth, facet_wrap, geom_violin, theme, element_blank, geom_text, geom_bar, geom_hline
import plotnine
from tqdm import tqdm
plotnine.options.figure_size = (4, 4)
import seaborn as sns
sns.set_theme(style="whitegrid")

### Settings and Load data

In [ ]:
torch.manual_seed(42)

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

K = 50 # set to very high number 

float_type = { 
    "device" : device, 
    "dtype" : torch.float, # save memory
}

hypers = {
    "eta" : 1./K, 
    "alpha_prior" : 1., # karin had 0.65 
    "pi_prior" : 1.
}

print(hypers["eta"])

### load_cluster_data takes ~ 5 minutes for ss2 brain data.... 

In [ ]:
# this folder contains input data for each tissue cell type sample
input_files_folder = '/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/MLCB_Brain_trueANNOfree/train/'

final_data, coo_counts_sparse, coo_cluster_sparse, cell_ids_conversion, junction_ids_conversion = load_cluster_data.load_cluster_data(
    input_folder = input_files_folder, has_genes="no") 

In [ ]:
junction_ids_conversion.head()

In [ ]:
print("The number of junctions is: ", len(junction_ids_conversion))
print("The number of intron clusters observed is: ", len(junction_ids_conversion.Cluster.unique()))

In [ ]:
final_data.head()

In [ ]:
# ensure that in coo_counts_sparse.shape = (n_cells, n_genes) , n_cells is the same number as cell_ids_conversion.shape = (n_cells, num_variables)
assert coo_counts_sparse.shape[0] == cell_ids_conversion.shape[0]

In [ ]:
cell_index_tensor, junc_index_tensor, my_data = make_torch_data(final_data, **float_type)

In [ ]:
cell_ids_conversion.head()
# what is the breadkdown of cell types in the data 
cell_ids_conversion['cell_type'].value_counts()

In [ ]:
# set random seed
torch.manual_seed(0)

num_trials = 10 # should also be an argument that gets fed in
num_iters = 100 # should also be an argument that gets fed in
K = 10

# loop over the number of trials (for now just testing using one trial but in general need to evaluate how performance is affected by number of trials)
#reload(betabinomo_mix_singlecells)

start_time = time.time()

# Running for just one K and assessing similarity across trials 

results = [ calculate_CAVI(K, my_data, float_type, hypers, init_labels = None, num_iterations = num_iters) 
           for t in range(num_trials) ]


# write the above line use fstring
print(f"This took {time.time() - start_time} seconds")

In [ ]:
running_multiple_K = False 
if running_multiple_K:
    elbos_ks = []
    for i in range(len(all_results_k)):
        print("Reporting ELBO for k = " + str(i+1))
        results = all_results_k[i]
        best = np.argmax([ g[-1][-1] for g in results ]) # best ELBO within trial
        ALPHA_f, PI_f, GAMMA_f, PHI_f, elbos_all = results[best]
        elbos_all = np.array(elbos_all)
        print("ELBO: " + str(elbos_all[-1]))
        elbos_ks.append(elbos_all[-1])
    
    elbos_ks = pd.DataFrame(elbos_ks)
    elbos_ks["k"] = range(K)
    best_k = elbos_ks.sort_values(by=0, ascending=False).head(1)
    results = all_results_k[best_k.index[0]]
    print("The k with the highest ELBO is: " + str(best_k.index[0]+1))
    K = best_k.index[0]+1 
    print(K)

### Consensus Clustering

In [ ]:
sum_matrices = consensus_clustering(results)

# normalize by number of trials
normalized_matrix = sum_matrices / sum_matrices.max() # taking the sum_matrix and dividing by the max value in the matrix

# get distance metric 
distance_matrix = 1 - normalized_matrix
distance_matrix

In [ ]:
# what is the proportion of 0s and 1s in the distance matrix
print("Proportion of 0s in distance matrix: ", np.count_nonzero(distance_matrix == 0) / (distance_matrix.shape[0] * distance_matrix.shape[1]))
print("Proportion of 1s in distance matrix: ", np.count_nonzero(distance_matrix == 1) / (distance_matrix.shape[0] * distance_matrix.shape[1]))

In [ ]:
final_data.cell_type.unique()

In [ ]:
# Compute the variance for each cell (along rows)
variances_per_cell = np.var(distance_matrix, axis=1)

# Compute the overall measure of consistency
overall_consistency = np.mean(variances_per_cell)

print(overall_consistency)

In [ ]:
# cluster distance matrix using hierarchical clustering into K clusters 
from sklearn.cluster import KMeans
# Number of clusters
num_clusters = 9

# Perform K-means clustering
kmeans = KMeans(n_clusters=num_clusters)
cluster_labels = kmeans.fit_predict(distance_matrix)

# Your N by K matrix
n_by_k_matrix = np.zeros((distance_matrix.shape[0], num_clusters))

# Fill the N by K matrix with cluster assignments
for i in range(num_clusters):
    n_by_k_matrix[:, i] = (cluster_labels == i).astype(int)

In [ ]:
# make also a clustermap of sum_matrices along with their cell types 
# sample 5200 indices from distance matrix
samp_indices = np.random.choice(cell_ids_conversion.shape[0], 1000, replace=False)
cell_types_heatmap = cell_ids_conversion.iloc[samp_indices]

color_palette = sns.color_palette("Set1", n_colors=len(cell_types_heatmap['cell_type'].unique()))

# Create a color bar legend
legend = sns.color_palette(palette=color_palette, as_cmap=True)

# Obtain cell type labels for every cell in the matrix also 
unique_cell_types = cell_types_heatmap['cell_type'].unique()
num_unique_types = len(unique_cell_types)
colors = sns.color_palette('Set1', n_colors=num_unique_types)  # You can use any color palette
cell_type_colors = {cell_type: color for cell_type, color in zip(unique_cell_types, colors)}
cell_types = cell_types_heatmap.cell_type.values

# Convert cell types to corresponding colors for rows and columns
row_colors = [cell_type_colors[cell_type] for cell_type in cell_types]
col_colors = [cell_type_colors[cell_type] for cell_type in cell_types]

cluster = sns.clustermap(
    data=distance_matrix[samp_indices,:][:,samp_indices],
    method='complete',
    cmap="viridis",
    annot=False,
    fmt=".2f",
    xticklabels=False,
    yticklabels=False,
    figsize=(8, 8),
    center=0,
    row_colors=row_colors,  # Apply row colors
    col_colors=col_colors   # Apply column colors
    )

In [ ]:
# Add a custom legend for row cell types
# make size bigger plt.fig size
plt.figure(figsize=(10, 10))
legend_labels = cell_type_colors.keys()
legend_colors = cell_type_colors.values()
# make background white
sns.set_style("white")
# make font of legend larger 
sns.set(font_scale=1.5)
custom_legend = [plt.Line2D([0], [0], marker='o', color='w', label=label, markersize=10, markerfacecolor=color) for label, color in zip(legend_labels, legend_colors)]
plt.legend(handles=custom_legend, title='Cell Types')

In [ ]:
# sample 5200 indices from distance matrix
samp_indices = np.random.choice(cell_ids_conversion.shape[0], 5200, replace=False)
cell_types_heatmap = cell_ids_conversion.iloc[samp_indices]

color_palette = sns.color_palette("Set1", n_colors=len(cell_types_heatmap['cell_type'].unique()))

# Create a color bar legend
legend = sns.color_palette(palette=color_palette, as_cmap=True)

# Obtain cell type labels for every cell in the matrix also 
unique_cell_types = cell_types_heatmap['cell_type'].unique()
num_unique_types = len(unique_cell_types)
colors = sns.color_palette('Set1', n_colors=num_unique_types)  # You can use any color palette
cell_type_colors = {cell_type: color for cell_type, color in zip(unique_cell_types, colors)}
cell_types = cell_types_heatmap.cell_type.values

# Convert cell types to corresponding colors for rows and columns
row_colors = [cell_type_colors[cell_type] for cell_type in cell_types]
col_colors = [cell_type_colors[cell_type] for cell_type in cell_types]

In [ ]:
# draw clustermap of sampled_distance_matrix
# Create a clustermap with row colors representing cell types

clustmap = sns.clustermap(
    data=n_by_k_matrix[samp_indices,:],
    annot=False,
    fmt=".2f",
    xticklabels=True,
    yticklabels=False,
    row_colors=row_colors, 
    figsize=(8, 8))

# increase font size of xticklabels
clustmap.ax_heatmap.set_xticklabels(clustmap.ax_heatmap.get_xmajorticklabels(), fontsize = 16)



In [ ]:
# Add a custom legend for row cell types
# make size bigger plt.fig size
plt.figure(figsize=(10, 10))
legend_labels = cell_type_colors.keys()
legend_colors = cell_type_colors.values()
# make background white
sns.set_style("white")
# make font of legend larger 
sns.set(font_scale=1.5)
custom_legend = [plt.Line2D([0], [0], marker='o', color='w', label=label, markersize=10, markerfacecolor=color) for label, color in zip(legend_labels, legend_colors)]
plt.legend(handles=custom_legend, title='Cell Types')

### Evaluate the learned posteriors

In [ ]:
best = np.argmax([ g[-1][-1] for g in results ]) # final ELBO
print(f"The trial with the highest ELBO was {best}")
ALPHA_f, PI_f, GAMMA_f, PHI_f, elbos_all = results[best]
elbos_all = np.array(elbos_all)
plt.plot(elbos_all[1:]); plt.show()

In [ ]:
juncs_probs = ALPHA_f / (ALPHA_f+PI_f)   
 
plt.hist(juncs_probs.cpu().numpy().flatten(), 20)
plt.title('Histogram of learned junction probabilities') 
plt.xlabel('Probability of junction success')
plt.show()

In [ ]:
PHI_f_plot = pd.DataFrame(PHI_f.cpu().numpy())
PHI_f_plot['cell_id'] = cell_ids_conversion["cell_type"].to_numpy()

In [ ]:
# How much is each cell state is used globally 

# Calculate the total sum of the tensor values
total_sum = torch.sum(GAMMA_f)

# Calculate the percentages
percentages = (GAMMA_f / total_sum) 

# Convert the tensor to a dataframe 
GAMMA_f_plot = pd.DataFrame(percentages.cpu().numpy())
# Give it a colname called theta 
GAMMA_f_plot.columns = ["theta"]
GAMMA_f_plot["cell_state"] = GAMMA_f_plot.index
GAMMA_f_plot.sort_values(by="theta", ascending=False, inplace=True)
GAMMA_f_plot["new_cell_state"] = np.arange(GAMMA_f_plot.shape[0])

sorted_cell_states = GAMMA_f_plot["new_cell_state"].astype(str)

# rename cell_state to be from 0 to K-1 based on order in sorted_cell_states
GAMMA_f_plot.head()

GAMMA_f_plot["new_cell_state"] = pd.Categorical(sorted_cell_states, sorted_cell_states.unique())

# make barplot using sns.
plt.figure(figsize=(7, 6))
sns.barplot(x="new_cell_state", y="theta", data=GAMMA_f_plot)
plt.axhline(0.01, ls='--', color='red')
# make Y lab say Theta and increase font of all labels and ticks
plt.ylabel("Theta", fontsize=16)
plt.xlabel("Learned Cell States (K)", fontsize=16)
# remove xaxis ticks
plt.xticks([])
plt.yticks(fontsize=16)
plt.show()

In [ ]:
# let's retain only the cell states that are used more than 1% of the time
GAMMA_f_plot = GAMMA_f_plot[GAMMA_f_plot["theta"] > 0.01]
GAMMA_f_plot.index.values

In [ ]:
# convert PHI_f to a dataframe and add a column with cell ID and cell type 
PHI_f = pd.DataFrame(PHI_f)

# Keep only cell states defined above GAMMA_f_plot.index.values
PHI_f = PHI_f.loc[:, GAMMA_f_plot.index.values]
PHI_f

In [ ]:
# Add "CellState" to each column 
PHI_f.columns = ["CellState_" + str(i) for i in range(PHI_f.shape[1])]
PHI_f['cell_id'] = cell_ids_conversion.cell_id.values
PHI_f['cell_type'] = cell_ids_conversion.cell_type.values
PHI_f.groupby('cell_type').sum()

In [ ]:
PHI_f.shape # 29 cell states kept 

In [ ]:
# group by cell_type and sum across each cellstate 
PHI_f.groupby('cell_type').sum()
sum_prop=PHI_f.groupby('cell_type').sum()/PHI_f.groupby('cell_type').count()
# remove cell_id column 
sum_prop=sum_prop.drop(columns=['cell_id'])
#masked_data = np.ma.masked_equal(sum_prop, 0)
sns.set(font_scale=0.8)  # Adjust font size for labels
# make figure bigger 
plt.figure(figsize=(6, 6))
# make font size of xtickts and yticks bigger
plt.yticks(fontsize=10)
plt.xticks(fontsize=10)
sns.heatmap(sum_prop, annot=False, fmt=".2f", cmap='viridis')

# now let's give the cell states new labels based on what cell types they are associated with 
# go through sum_prop and for each cell state find the cell type with the highest proportion
# assign that cell type to the cell state
cell_type_labels = {}
for state in sum_prop.columns:
    max_prop = 0
    max_celltype = ''
    for celltype in sum_prop.index:
        if sum_prop.loc[celltype, state] > max_prop:
            max_prop = sum_prop.loc[celltype, state]
            max_celltype = celltype
    cell_type_labels[state] = max_celltype, round(max_prop, 3)
print(cell_type_labels)

## Do differential junction analysis 

In [ ]:
juncs_probs = juncs_probs[:, GAMMA_f_plot.index.values]

In [ ]:
juncs_probs_df = pd.DataFrame(juncs_probs)
# add "cell_state" to each column name 
juncs_probs_df.columns = ["cell_state_" + str(col) for col in juncs_probs_df.columns]
juncs_probs_df["junction_id_index"] = junction_ids_conversion.junction_id_index.values
# convert to juncs_probs to pandas dataframe and calculate mean and std across cell states/topics
juncs_probs_df["junction_id"] = junction_ids_conversion.junction_id.values
juncs_probs_df

In [ ]:
# get likelihood ratio/bayes factor score for ALL junctions 
# let's compare just state X and Y

scores_all_juncs = []
for junc_index in tqdm(range(juncs_probs.shape[0])):
    a = ALPHA_f[junc_index, ]
    b = PI_f[junc_index, ]
    scores_all_juncs.append(score(a, b).item())

# turn scores_all_juncs into dataframe and add junction_id_index as a column
scores_all_juncs_df = pd.DataFrame(scores_all_juncs, columns = ["score"])
scores_all_juncs_df["junction_id_index"] = junction_ids_conversion.junction_id_index.values
scores_all_juncs_df.sort_values(by="score", ascending=False)
juncs_test=scores_all_juncs_df.sort_values(by="score", ascending=False).head(2).junction_id_index.values

In [ ]:
scores_all_juncs_df = scores_all_juncs_df.merge(junction_ids_conversion, on="junction_id_index").sort_values(by="score", ascending=False)

In [ ]:
scores_all_juncs_df

In [ ]:
# calculate the number of junctions in each cluster and add that on as a column to scores_all_juncs_df
num_juncs_per_cluster = scores_all_juncs_df.groupby("Cluster")["junction_id_index"].count()
num_juncs_per_cluster = num_juncs_per_cluster.to_frame().reset_index()
num_juncs_per_cluster.columns = ["Cluster", "num_juncs"]

# merge num_juncs_per_cluster with scores_all_juncs_df
scores_all_juncs_df = scores_all_juncs_df.merge(num_juncs_per_cluster, on="Cluster")
scores_all_juncs_df.head()

In [ ]:
scores_all_juncs_df[scores_all_juncs_df["Cluster"] == 4646]

In [ ]:
test_dat = simple_data[simple_data["Cluster"] == 13103].sort_values(by="junc_ratio", ascending=False)
# report min, max, median and mean juncratio for each cell type
test_dat.groupby("cell_type")["junc_ratio"].describe()

In [ ]:
quick_clust_plot(13103, sim)

In [ ]:
test_dat.junction_id.unique()

In [ ]:
print(quick_junc_plot(43026, simple_data))
print(quick_junc_plot(43027, simple_data))
print(quick_junc_plot(42995, simple_data))

In [ ]:
scores_all_juncs_df.sort_values(by="score", ascending=False).head(50)

In [ ]:
quick_clust_plot(13103, simple_data)

In [ ]:
def quick_junc_plot(junc, simple_data, gene_name=None):
    simple_data_junc = simple_data[simple_data["junction_id_index"] == junc]
    # make violin plot with jitter 
    print(simple_data_junc.cell_type.value_counts())

    sns.violinplot(data = simple_data_junc, x = "junc_ratio", y = "cell_type")
    # make xlim -1 to 1.1
    plt.xlim(-0.2, 1.2)
    # add sample_label to title 
    if gene_name:
        plt.title("Junction:" + str(junc) + " Gene: " + simple_data_junc["gene_id"].values[0])
    # set x axis label to "Junction Usage Ratio (PSI)"
    plt.xlabel("Junction Usage Ratio (PSI)")
    plt.show()

In [ ]:
def quick_clust_plot(clust, simple_data, gene_name=None):
    simple_data_junc = simple_data[simple_data["Cluster"] == clust]
    # make violin plot with jitter 
    print(simple_data_junc.cell_type.value_counts())

    # set figure size to be 6 by 7 
    plt.figure(figsize=(8, 8))
    sns.violinplot(data = simple_data_junc, x = "junc_ratio", y = "cell_type", hue="junction_id_index")
    # make xlim -1 to 1.1
    plt.xlim(-0.2, 1.2)
    # add sample_label to title 
    if gene_name:
        plt.title("Cluster:" + str(clust) + " Gene: " + simple_data_junc["gene_id"].values[0], fontsize=16)
    # set x axis label to "Junction Usage Ratio (PSI)"
    # increase font size of title and xaxis and yaxis labels
    plt.xlabel("Junction Usage Ratio (PSI)", fontsize=16)
    plt.ylabel("Cell Type", fontsize=16)
    # increase font size of xticks and yticks
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.show()

In [ ]:
simple_data = final_data[["cell_id_index", "cell_type", "junction_id_index", "junc_count", "cluster_count"]]
simple_data = simple_data.merge(junction_ids_conversion, on="junction_id_index")
simple_data["junc_ratio"] = simple_data["junc_count"] / simple_data["cluster_count"]
simple_data.head()

In [ ]:
scores_all_juncs_df

In [ ]:
quick_junc_plot(395, simple_data, False)

In [ ]:
# subset ALPHA_f and PI_f to just states in GAMMA_f_plot.index.values
ALPHA_f = ALPHA_f[:, GAMMA_f_plot.index.values]
PI_f = PI_f[:, GAMMA_f_plot.index.values]

In [ ]:
# make a version of the plot that shows that learned junction proportions across cell states 
from scipy.stats import beta

def quick_posterior_junc_plot(junc, ALPHAs, PIs):
    
    junc_a = ALPHAs[junc, ]
    junc_b = PIs[junc, ]

    # Define a list of colors for the plot
    colors = ['r', 'g', 'b', 'c', 'm', 'y', 'k']  # Add more colors if needed

    # Create a figure and axis for the plot
    fig, ax = plt.subplots()

    print(junc_a, junc_b)
    
    # Loop through each cell state and plot its beta distribution with a unique color
    for j, (a, b) in enumerate(zip(junc_a, junc_b)):
        
        x = np.linspace(beta.ppf(0.01, a, b),
                        beta.ppf(0.99, a, b), 100)
        
        color = colors[j % len(colors)]  # Cycle through colors if more cell states than colors
        label = f'Cell State {j + 1}'
        
        ax.plot(x, beta.pdf(x, a, b),
                color + '-', lw=2, alpha=0.6, label=label)
        
    # Add labels and legend
    ax.set_xlabel('Probability of success')
    ax.set_ylabel('Probability Density')
    ax.set_title('Beta Distributions for Different Cell States in junction ' + str(junc))
    
    # Place the legend to the right of the plot
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.show()


In [ ]:
quick_posterior_junc_plot(1000, ALPHA_f, PI_f)

In [ ]:
quick_junc_plot(2000, simple_data)

In [ ]:
# save cell assignments to file for downstream analysis
output_dir = '/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/'

# BBmixture model cell assignments
output_file = os.path.join(output_dir, 'Leaflet_BBmixture.csv')
PHI_f.to_csv(output_file, index=True, header=True)
print('Saved Leaflet latent cell states to {}'.format(output_file))

In [ ]:
# Let's find the junctions that are the most differentially spliced between microglia and astrocytes 
scores_all_juncs = []
cellstate1 = 0
cellstate2 = 1

for junc_index in range(juncs_probs.shape[0]):
    a = ALPHA_f[junc_index, [cellstate1,cellstate2]]
    b = PI_f[junc_index, [cellstate1,cellstate2]]
    scores_all_juncs.append(score(a, b).item())

# turn scores_all_juncs into dataframe and add junction_id_index as a column
scores_all_juncs_df = pd.DataFrame(scores_all_juncs, columns = ["score"])
scores_all_juncs_df["junction_id_index"] = junction_ids_conversion.junction_id_index.values
scores_all_juncs_df.sort_values(by="score", ascending=False).head(10)
juncs_test=scores_all_juncs_df.sort_values(by="score", ascending=False).head(2).junction_id_index.values

## Evaluate likelihood on validation set to determine best K

In [ ]:
# validation data 
input_files_folder = '/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/MLCB_Brain_true/validation/'

val_data, val_coo_counts_sparse, val_coo_cluster_sparse, val_cell_ids_conversion, val_junction_ids_conversion = load_cluster_data.load_cluster_data(
    input_folder = input_files_folder, has_genes="yes")
cell_index_tensor_val, junc_index_tensor_val, my_data_val = make_torch_data(val_data, **float_type)

In [ ]:
import betabinomo_mix_singlecells
reload(betabinomo_mix_singlecells)

In [ ]:
# set random seed
torch.manual_seed(0)

num_trials = 1 # should also be an argument that gets fed in
num_iters = 40 # should also be an argument that gets fed in
K = 12

# loop over the number of trials (for now just testing using one trial but in general need to evaluate how performance is affected by number of trials)
#reload(betabinomo_mix_singlecells)

start_time = time.time()
all_results_k = []
all_ll_k = []

for k in range(K):
    k = k + 1
    print(f"Running with {k} cell states")
    results = [ betabinomo_mix_singlecells.calculate_CAVI(k, my_data, float_type, hypers, init_labels = None, num_iterations = num_iters) 
           for t in range(num_trials) ]
    
    all_results_k.append(results)
    ALPHA_f, PI_f, GAMMA_f, PHI_f, elbos_all = results[0]
    juncs_probs = ALPHA_f / (ALPHA_f+PI_f)   
    juncs_probs_df = pd.DataFrame(juncs_probs.numpy())
    # rename columns to be "state_0", "state_1", etc.
    juncs_probs_df.columns = ["state_" + str(i) for i in range(juncs_probs_df.shape[1])]
    juncs_probs_df["junction_id_index"] = juncs_probs_df.index

    # merge with val_data 
    val_data_run = val_data.merge(juncs_probs_df, on="junction_id_index")
    theta = GAMMA_f / GAMMA_f.sum()
    theta = theta.cpu().numpy()

    # evaluate predictive log likelihood on validation set 
    print("Calculating predictive log likelihood on validation set")
    ll = betabinomo_mix_singlecells.calculate_predictive_lik(theta, val_data_run)
    all_ll_k.append(ll)
    print(f"Predictive log likelihood on validation set is {ll}")

# write the above line use fstring
print(f"This took {time.time() - start_time} seconds")

In [ ]:
(all_ll_k == np.max(all_ll_k))

In [ ]:
all_ks = pd.DataFrame(np.array(all_ll_k))
all_ks["k"] = all_ks.index
all_ks.columns = ["ll", "k"]
all_ks.sort_values("ll", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

# Given data
data = [
    (10, -3.759537e+07),
    (11, -3.792445e+07),
    (9, -3.812037e+07),
    (7, -3.837194e+07),
    (8, -3.856408e+07),
    (6, -3.934490e+07),
    (5, -3.983091e+07),
    (4, -4.040786e+07),
    (2, -4.199341e+07),
    (3, -4.218076e+07),
    (1, -4.448278e+07),
    (0, -4.958535e+07)
]

# Unpack data into separate lists for x and y values
x_values, y_values = zip(*data)

# Create a bar plot
plt.bar(x_values, y_values, color='blue')

# Add labels and title
plt.xlabel('k', fontsize=14)
plt.ylabel('Log Likelihood (ll)', fontsize=14)
plt.title('Log Likelihood vs k', fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

# Show the plot
plt.show()
